# 04 — Model Registration and Serving Endpoint Setup
---
**What this notebook does:**
This is the final step in the Databricks pipeline. It:
1. Promotes both trained models to **Production** stage in the MLflow Model Registry
2. Walks you through creating the two **Databricks Serving Endpoints** via the UI
3. Tests both endpoints with live REST API calls to confirm they return predictions
4. Prints the exact `.env` values you need to paste into the Flask app

**Prerequisites:**
- Notebook 01 must have run (Delta table exists)
- Notebook 02 must have run (`ohs_severity_classifier` registered in MLflow)
- Notebook 03 must have run (`ohs_risk_score_model` registered in MLflow)
- Both serving endpoints must be created in the UI before running the test cells

**Runtime:** Serverless &nbsp;|&nbsp; **No additional pip installs required**

## Cell 1 — Imports

In [0]:
import mlflow                    # for MLflow Model Registry operations
import requests                  # for making REST API calls to the serving endpoints
import json                      # for pretty-printing JSON responses
from mlflow.tracking import MlflowClient

# MlflowClient gives us programmatic access to the Model Registry
# (create, update, and transition model versions)
client = MlflowClient()

## Cell 2 — Promote Both Models to Production Stage

MLflow Model Registry has four stages: `None`, `Staging`, `Production`, `Archived`.
We promote both models to **Production** so the serving endpoints always point
to the latest approved version, and older versions are automatically archived.

**What `archive_existing_versions=True` does:**
Any version of this model currently in Production is automatically moved to
Archived — so there is always exactly one Production version at a time.

After running this cell, go to **Machine Learning → Models** and confirm
both models show a green **Production** badge on Version 1.

In [0]:
def promote_to_production(model_name: str, version: int | None = None) -> int:
    """
    Sets the 'production' alias on the specified (or latest) model version.
    Unity Catalog uses aliases instead of stages for model deployment.

    Parameters:
      model_name : the registered model name in MLflow (e.g. 'workspace.default.ohs_severity_classifier')
      version    : the version number to promote; if None, promotes the latest version

    Returns:
      the version number that was promoted
    """
    if version is None:
        # Find the latest version number by searching all registered versions
        all_versions = client.search_model_versions(f"name='{model_name}'")
        version = max(int(v.version) for v in all_versions)

    # Set the 'production' alias on this version
    # This automatically moves the alias from any previous version
    client.set_registered_model_alias(
        name=model_name,
        alias="production",
        version=version,
    )
    print(f"✓  {model_name}  v{version}  →  production alias set")
    return version


# Promote both models — use three-level Unity Catalog names
sev_ver  = promote_to_production("workspace.default.ohs_severity_classifier")
risk_ver = promote_to_production("workspace.default.ohs_risk_score_model")

print(f"\nBoth models now have the 'production' alias.")
print(f"  ohs_severity_classifier  → Version {sev_ver}")
print(f"  ohs_risk_score_model     → Version {risk_ver}")

✓  workspace.default.ohs_severity_classifier  v1  →  production alias set
✓  workspace.default.ohs_risk_score_model  v1  →  production alias set

Both models now have the 'production' alias.
  ohs_severity_classifier  → Version 1
  ohs_risk_score_model     → Version 1


## Step 2 — Create Serving Endpoints in the Databricks UI
---

In the left sidebar click **Serving** → **Create serving endpoint**

### Endpoint 1 — NLP Severity Classifier
| Field | Value |
|---|---|
| Endpoint name | `ohs-severity-endpoint` |
| Served entity | `ohs_severity_classifier` |
| Version | 1 (the version just promoted to Production) |
| Compute size | Small |
| Scale to zero | Enabled |

### Endpoint 2 — Risk Score Model
| Field | Value |
|---|---|
| Endpoint name | `ohs-risk-score-endpoint` |
| Served entity | `ohs_risk_score_model` |
| Version | 1 (the version just promoted to Production) |
| Compute size | Small |
| Scale to zero | Enabled |

> **Important:** Wait until BOTH endpoints show **Ready** (green) before
> running the test cells below. This usually takes 5–10 minutes.
> Scale-to-zero means the first request after idle takes ~30–60 s to warm up.

## Cell 3 — Set Your Credentials

**How to get your PAT (Personal Access Token):**
1. Click your profile avatar (top-right corner)
2. Go to **Settings → Developer → Access tokens**
3. Click **Generate new token**, give it a name, click **Generate**
4. Copy the token — **it is only shown once**

**How to get your workspace URL:**
- It is in your browser address bar: `https://xxxxxxx.azuredatabricks.net`

Paste both values into the two variables below.

In [0]:
# ── PASTE YOUR VALUES HERE ─────────────────────────────────────────────────────
# dbutils.secrets is NOT available on the free Databricks serverless tier.
# Enter your credentials directly as strings for this POC notebook.
# Do NOT commit this notebook to Git with real credentials inside.

DATABRICKS_HOST  = "https://dbc-71dc0835-779f.cloud.databricks.com"   # ← paste your workspace URL
DATABRICKS_TOKEN = "Paste your PAT code"          # ← paste your PAT

# Build the serving endpoint URLs from your workspace host
# These URLs follow the standard Databricks Model Serving invocation pattern
SEVERITY_URL = f"{DATABRICKS_HOST}/serving-endpoints/ohs-severity-endpoint/invocations"
RISK_URL     = f"{DATABRICKS_HOST}/serving-endpoints/ohs-risk-score-endpoint/invocations"

# All REST API calls to Databricks require Bearer token authentication
HEADERS = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type":  "application/json",
}

print("Endpoint URLs configured:")
print(f"  Severity : {SEVERITY_URL}")
print(f"  Risk     : {RISK_URL}")

Endpoint URLs configured:
  Severity : https://dbc-71dc0835-779f.cloud.databricks.com/serving-endpoints/ohs-severity-endpoint/invocations
  Risk     : https://dbc-71dc0835-779f.cloud.databricks.com/serving-endpoints/ohs-risk-score-endpoint/invocations


## Cell 3.5 — Create Both Serving Endpoints Programmatically

Instead of manually creating endpoints in the UI (as described in Cell 6),
this cell creates both endpoints via the Databricks REST API.

**What this does:**
- Creates `ohs-severity-endpoint` serving the severity classifier model
- Creates `ohs-risk-score-endpoint` serving the risk score model
- Both use Small compute with scale-to-zero enabled
- Endpoints take 5–10 minutes to become Ready

**After running:** Skip to Cell 4 (test cells) once both endpoints show Ready status.

In [0]:
import time

# Databricks Serving Endpoints API base URL
API_BASE = f"{DATABRICKS_HOST}/api/2.0/serving-endpoints"

def create_serving_endpoint(endpoint_name: str, model_name: str, model_version: int):
    """
    Creates a Databricks Model Serving endpoint with Small compute and scale-to-zero.
    
    Parameters:
      endpoint_name  : the serving endpoint name (e.g. 'ohs-severity-endpoint')
      model_name     : the registered model name (e.g. 'workspace.default.ohs_severity_classifier')
      model_version  : the model version to serve (e.g. 1)
    """
    config = {
        "name": endpoint_name,
        "config": {
            "served_entities": [
                {
                    "entity_name": model_name,
                    "entity_version": str(model_version),
                    "workload_size": "Small",
                    "scale_to_zero_enabled": True,
                }
            ]
        }
    }
    
    # Check if endpoint already exists
    check_resp = requests.get(
        f"{API_BASE}/{endpoint_name}",
        headers=HEADERS,
    )
    
    if check_resp.status_code == 200:
        print(f"⚠  Endpoint '{endpoint_name}' already exists — skipping creation")
        return
    
    # Create the endpoint
    resp = requests.post(
        API_BASE,
        headers=HEADERS,
        json=config,
    )
    
    if resp.status_code in [200, 201]:
        print(f"✓  Created endpoint: {endpoint_name}")
        print(f"   Model: {model_name} v{model_version}")
    else:
        print(f"✗  Failed to create {endpoint_name}")
        print(f"   Status: {resp.status_code}")
        print(f"   Response: {resp.text}")


# Create both endpoints
print("Creating serving endpoints...\n")

create_serving_endpoint(
    endpoint_name="ohs-severity-endpoint",
    model_name="workspace.default.ohs_severity_classifier",
    model_version=1,
)

print()

create_serving_endpoint(
    endpoint_name="ohs-risk-score-endpoint",
    model_name="workspace.default.ohs_risk_score_model",
    model_version=1,
)

print("\n" + "="*70)
print("  Both endpoints are now being created.")
print("  This takes 5–10 minutes. Check status in Serving UI or wait below.")
print("="*70)

Creating serving endpoints...

✓  Created endpoint: ohs-severity-endpoint
   Model: workspace.default.ohs_severity_classifier v1

✓  Created endpoint: ohs-risk-score-endpoint
   Model: workspace.default.ohs_risk_score_model v1

  Both endpoints are now being created.
  This takes 5–10 minutes. Check status in Serving UI or wait below.


## Cell 4 — Test the Severity Endpoint

Sends a single inspection text to the NLP severity endpoint and prints the response.

**Databricks serving payload format:**
```json
{"dataframe_records": [{"inspection_text": "..."}]}
```
The `dataframe_records` key is required by Databricks Model Serving when
the model expects a pandas DataFrame input.

**Expected response:**
```json
{"predictions": [{"severity_level": "Critical", "confidence": 0.92, "probabilities": {...}}]}
```

**If you get HTTP 503:** the endpoint is warming up from scale-to-zero — wait 60 seconds and retry.
**If you get HTTP 404:** the endpoint name doesn't match — check the name in the Serving UI.

In [0]:
# Test payload — a single high-severity inspection text
severity_payload = {
    "dataframe_records": [
        {
            "inspection_text": (
                "Worker sustained critical injuries from unguarded rotating machinery. "
                "No lockout tagout procedures were in place at the workplace. "
                "Stop Work Order issued immediately pending full investigation."
            )
        }
    ]
}

print("Sending request to severity endpoint...")
print(f"URL: {SEVERITY_URL}\n")

resp = requests.post(
    SEVERITY_URL,
    headers=HEADERS,
    json=severity_payload,
    timeout=180,    # increased timeout to 180 seconds to prevent read timeout errors
)

print(f"HTTP Status: {resp.status_code}")
print("\nFull response:")
print(json.dumps(resp.json(), indent=2))

# Quick assertion — confirm we got a severity_level back
if resp.status_code == 200:
    prediction = resp.json().get("predictions", [{}])[0]
    print(f"\n✓  Severity predicted: {prediction.get('severity_level')}  "
          f"(confidence: {prediction.get('confidence', 0):.1%})")
else:
    print(f"\n✗  Request failed — see response above for details")

Sending request to severity endpoint...
URL: https://dbc-71dc0835-779f.cloud.databricks.com/serving-endpoints/ohs-severity-endpoint/invocations

HTTP Status: 200

Full response:
{
  "predictions": [
    {
      "severity_level": "High",
      "confidence": 0.2981,
      "probabilities": {
        "Critical": 0.2272,
        "High": 0.2981,
        "Low": 0.2671,
        "Medium": 0.2076
      }
    }
  ]
}

✓  Severity predicted: High  (confidence: 29.8%)


## Cell 5 — Test the Risk Score Endpoint

Sends a full structured inspection record to the risk score endpoint.
Note that `SEVERITY_LEVEL` is included in the payload — in the real
Flask app pipeline this value comes from the severity endpoint's output
(Model 1 feeds Model 2).

**Expected response:**
```json
{"predictions": [{"risk_score": 91.4, "risk_category": "Critical Risk", "risk_level_int": 4}]}
```

In [0]:
# Test payload — full structured record including SEVERITY_LEVEL from Model 1
risk_payload = {
    "dataframe_records": [{
        "FIELD_VISIT_DATE":  "2024-03-15",
        "FIELD_VISIT_TYPE":  "Field Visit",
        "CASE_TYPE":         "Investigation",        # reactive — incident happened
        "CASE_STATUS":       "Open",
        "PRIMARY_NAICS":    "236110",               # Residential Construction
        "CONTRAVENER_ROLE":  "Constructor",
        "ORDER_TYPE":        "Stop Work Order",      # most severe order type
        "ORDER_STATUS":      "Not Complied With",    # employer hasn't fixed it yet
        "CASE_ACT":          "Occupational Health and Safety Act",
        "ACT_REG_ID":        "REG_213",             # Construction Projects regulation
        "SEC":               "25",
        "SUBSEC":            "(1)",
        "CLAUSE":            "(a)",
        "SEVERITY_LEVEL":    "Critical",            # ← this comes from Model 1 output
    }]
}

print("Sending request to risk score endpoint...")
print(f"URL: {RISK_URL}\n")

resp = requests.post(
    RISK_URL,
    headers=HEADERS,
    json=risk_payload,
    timeout=90,
)

print(f"HTTP Status: {resp.status_code}")
print("\nFull response:")
print(json.dumps(resp.json(), indent=2))

# Quick assertion — confirm we got a risk_score back
if resp.status_code == 200:
    prediction = resp.json().get("predictions", [{}])[0]
    print(f"\n✓  Risk score predicted: {prediction.get('risk_score')} / 100  "
          f"→  {prediction.get('risk_category')}")
else:
    print(f"\n✗  Request failed — see response above for details")

Sending request to risk score endpoint...
URL: https://dbc-71dc0835-779f.cloud.databricks.com/serving-endpoints/ohs-risk-score-endpoint/invocations

HTTP Status: 200

Full response:
{
  "predictions": [
    {
      "risk_score": 98.0,
      "risk_category": "Critical Risk",
      "risk_level_int": 4
    }
  ]
}

✓  Risk score predicted: 98.0 / 100  →  Critical Risk


## Cell 6 — Print Flask App `.env` Configuration

Once both endpoints return HTTP 200, copy the output below and paste it
into the `.env` file in your `ohs_prediction_app/` folder.
Then change `USE_MOCK_PREDICTIONS=False` so the Flask app calls the real endpoints.

In [0]:
print("=" * 65)
print("  COPY THESE VALUES INTO ohs_prediction_app/.env")
print("=" * 65)
print()
print(f"DATABRICKS_HOST={DATABRICKS_HOST}")
print(f"DATABRICKS_TOKEN=<paste your PAT here — do not hardcode in .env if sharing>")
print(f"SEVERITY_ENDPOINT_URL={SEVERITY_URL}")
print(f"RISK_ENDPOINT_URL={RISK_URL}")
print(f"USE_MOCK_PREDICTIONS=False")
print(f"REQUEST_TIMEOUT=90")
print()
print("=" * 65)
print()
print("After updating .env, restart the Flask app:")
print("  python run.py")
print()
print("Then open http://localhost:5000, fill in the form, and confirm")
print("both models return predictions.")

  COPY THESE VALUES INTO ohs_prediction_app/.env

DATABRICKS_HOST=https://dbc-71dc0835-779f.cloud.databricks.com
DATABRICKS_TOKEN=<paste your PAT here — do not hardcode in .env if sharing>
SEVERITY_ENDPOINT_URL=https://dbc-71dc0835-779f.cloud.databricks.com/serving-endpoints/ohs-severity-endpoint/invocations
RISK_ENDPOINT_URL=https://dbc-71dc0835-779f.cloud.databricks.com/serving-endpoints/ohs-risk-score-endpoint/invocations
USE_MOCK_PREDICTIONS=False
REQUEST_TIMEOUT=90


After updating .env, restart the Flask app:
  python run.py

Then open http://localhost:5000, fill in the form, and confirm
both models return predictions.
